# African SLM Tool Calling Hackathon: starter notebook

Three approaches with **Gemma 3 270M**, each scored by exact-match accuracy:

1. **Zero-shot prompting**: no training.
2. **Fine-tuning** on 1,000 training examples.
3. **More training data**: the same fine-tuning on 3,000 examples.

Runtime: about 20 minutes on a Colab T4 GPU (*Runtime → Change runtime type → T4 GPU*).

# Install Libraries

In [ ]:
import importlib.util
import subprocess
import sys

missing = [package for package in ("peft", "accelerate") if importlib.util.find_spec(package) is None]
if missing:  # Colab already has torch and transformers
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("installed:", missing or "nothing needed")

# Environment and Imports

In [ ]:
import json
import math
import os
import random
import re
import time
import urllib.request
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda", torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.bfloat16
else:
    DEVICE, DTYPE = "cpu", torch.float32
print("device:", DEVICE)

# Charts: one fixed colour per approach.
BLUE, ORANGE, MUTED = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e4e3de", "axes.axisbelow": True, "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "legend.frameon": False})

Gemma needs a Hugging Face token from an account that has accepted the
[Gemma licence](https://huggingface.co/google/gemma-3-270m-it). On Colab, add it to *Secrets* as `HF_TOKEN`.

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login

    login(userdata.get("HF_TOKEN"))
except ImportError:  # not on Colab: use `huggingface-cli login` or the HF_TOKEN variable
    pass

The data is downloaded from the [hackathon repository](https://github.com/aims-ai-research-foundations/airf-slm-tool-calling-hackathon) into `data/`.

In [ ]:
REPO = "https://raw.githubusercontent.com/aims-ai-research-foundations/airf-slm-tool-calling-hackathon/main/data"
DATA_DIR = Path(os.environ.get("HACKATHON_DATA_DIR", "data"))
DATA_DIR.mkdir(exist_ok=True)
for name in ("tools.json", "train.jsonl", "dev.jsonl"):
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{REPO}/{name}", DATA_DIR / name)


def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def answer_of(row: dict):
    """A training row's answer: its tool_calls, or its tool and parameters."""
    return row["tool_calls"] if "tool_calls" in row else {"tool": row["tool"], "parameters": row["parameters"]}


def calls_of(answer) -> list[dict]:
    return answer if isinstance(answer, list) else [answer]


tools = {tool["name"]: tool for tool in json.loads((DATA_DIR / "tools.json").read_text())}
train = [dict(row, answer=answer_of(row)) for row in read_jsonl(DATA_DIR / "train.jsonl")]
dev = read_jsonl(DATA_DIR / "dev.jsonl")
print(f"{len(tools)} tools · {len(train):,} train · {len(dev):,} dev")

# Exploratory Data Analysis

In [ ]:
meta = pd.DataFrame([ex["meta"] for ex in train])
per_tool = Counter(call["tool"] for ex in train for call in calls_of(ex["answer"]))
nulls = Counter(param for params in meta["null_params"] for param in params)
panels = [
    ("Category", meta["category"].value_counts(normalize=True).sort_values()),
    ("Tools called", (pd.Series(per_tool) / sum(per_tool.values())).sort_values()),
    ("Country", meta["country"].value_counts(normalize=True).sort_values()),
    ("Most often null (share of the tool's calls)", pd.Series({p: n / per_tool[p.split(".")[0]] for p, n in nulls.items()}).sort_values().tail(8)),
]

fig, axes = plt.subplots(1, 4, figsize=(19, 4))
for ax, (title, values) in zip(axes, panels):
    ax.barh(values.index, values.values, color=ORANGE if "null" in title else BLUE, height=0.7)
    for y, v in enumerate(values.values):
        ax.text(v, y, f" {v:.0%}", va="center", fontsize=8)
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
    ax.grid(axis="y", visible=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
for category in ("single", "missing_info", "multiple"):
    ex = next(ex for ex in train if ex["meta"]["category"] == category)
    print(f"[{category}] {ex['request']}\n  -> {json.dumps(ex['answer'], ensure_ascii=False)}\n")

- 60% of requests need one call with every value given, 20% leave a value out (`null`), 20% need several calls.
- Tools and countries are evenly spread.
- The value most often missing is the currency: it is only given when the request writes it ("5k" alone has none).

# Gemma 270M Zero-Shot Prompting

The prompt is a one-line instruction, a signature for each offered tool, and the request. The model answers with
a JSON **list** of calls (a list of one call scores the same as the call itself). We score on **300 training
examples held out for validation**, 100 per category.

In [ ]:
MODEL_NAME = "google/gemma-3-270m-it"
INSTRUCTION = "Reply with a JSON list of the tool calls for the request, with null for values not given.\nTools:\n"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"


def build_prompt(example: dict) -> str:
    signatures = [f"{name}({', '.join(f'{p}: {t}' for p, t in tools[name]['parameters'].items())})" for name in example["tools"]]
    messages = [{"role": "system", "content": INSTRUCTION + "\n".join(signatures)}, {"role": "user", "content": example["request"]}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def target_text(answer) -> str:
    return json.dumps(calls_of(answer), ensure_ascii=False) + "<end_of_turn>"


by_category = defaultdict(list)
for ex in train:
    by_category[ex["meta"]["category"]].append(ex)
validation = [ex for group in by_category.values() for ex in random.Random(SEED).sample(group, 100)]
held = {ex["id"] for ex in validation}
train_pool = [ex for ex in train if ex["id"] not in held]

print(build_prompt(validation[0]) + target_text(validation[0]["answer"]))

In [ ]:
def normalise(value):
    """As the official scorer: numbers as floats; text lowercased without punctuation."""
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    if isinstance(value, str):
        return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", value.lower())).strip()
    return value


def canonical(answer):
    """A comparable form of an answer, with null parameters dropped; None if it is not a call or a list of calls."""
    try:
        return tuple((c["tool"], tuple(sorted((k, normalise(v)) for k, v in c["parameters"].items() if v is not None)))
                     for c in calls_of(answer))
    except (TypeError, KeyError, AttributeError):
        return None


def score(predictions: dict) -> dict:
    """Exact match on the validation set, overall and per category."""
    hits = defaultdict(list)
    for ex in validation:
        hits[ex["meta"]["category"]].append(canonical(predictions.get(ex["id"])) == canonical(ex["answer"]))
    return {"overall": np.mean(sum(hits.values(), []))} | {k: np.mean(hits[k]) for k in ("single", "missing_info", "multiple")}


def parse(text: str):
    """The JSON answer in the model's text, or None."""
    text = text.split("<end_of_turn>")[0]
    starts = [i for i in (text.find("["), text.find("{")) if i >= 0]
    try:
        value = json.JSONDecoder().raw_decode(text, min(starts))[0] if starts else None
    except json.JSONDecodeError:
        return None
    return value[0] if isinstance(value, list) and len(value) == 1 else value


@torch.no_grad()
def predict(model, examples: list[dict], batch_size: int = 32) -> dict:
    """Greedy answers, parsed: {id: answer or None}."""
    model.eval()
    order = sorted(examples, key=lambda ex: len(build_prompt(ex)))  # similar lengths batch together
    stop = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<end_of_turn>")]
    out = {}
    for i in range(0, len(order), batch_size):
        batch = order[i:i + batch_size]
        inputs = tokenizer([build_prompt(ex) for ex in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        tokens = model.generate(**inputs, max_new_tokens=200, do_sample=False, eos_token_id=stop, pad_token_id=tokenizer.pad_token_id)
        for ex, seq in zip(batch, tokens[:, inputs["input_ids"].shape[1]:]):
            out[ex["id"]] = parse(tokenizer.decode(seq, skip_special_tokens=False))
    return out

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, attn_implementation="eager").to(DEVICE)
zero_shot = predict(model, validation)
results = {"1. Zero-shot": score(zero_shot)}

ex = validation[0]
print("Request: ", ex["request"])
print("Expected:", json.dumps(ex["answer"], ensure_ascii=False))
print("Got:     ", zero_shot[ex["id"]])
pd.DataFrame(results).T.round(3)

Without training, Gemma 3 270M does not produce correct tool calls: its exact-match accuracy is close to zero.

# Gemma 270M Fine-Tuning

LoRA (rank 16 on every linear layer), learning rate 2e-4, one epoch on **1,000** training examples. The loss is
computed on the answer tokens only.

In [ ]:
def fine_tune(examples: list[dict], lr: float = 2e-4, batch_tokens: int = 4096):
    """A fresh Gemma 3 270M with a LoRA adapter, trained for one epoch; returns the model and the loss of each step."""
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, attn_implementation="eager").to(DEVICE)
    model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", r=16, lora_alpha=32, target_modules="all-linear"))
    for p in model.parameters():  # the adapter trains in float32 for stability
        if p.requires_grad:
            p.data = p.data.float()

    rows = []  # (token ids, labels): the loss is only on the answer
    for ex in examples:
        prompt = tokenizer(build_prompt(ex), add_special_tokens=False)["input_ids"]
        answer = tokenizer(target_text(ex["answer"]), add_special_tokens=False)["input_ids"]
        rows.append((prompt + answer, [-100] * len(prompt) + answer))
    random.Random(SEED).shuffle(rows)
    rows.sort(key=lambda row: len(row[0]) // 32)  # similar lengths in a batch
    batches, batch = [], []
    for row in rows:
        if batch and (len(batch) + 1) * len(row[0]) > batch_tokens:
            batches.append(batch)
            batch = []
        batch.append(row)
    batches.append(batch)
    random.Random(SEED).shuffle(batches)

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    schedule = torch.optim.lr_scheduler.LambdaLR(  # short warm-up, then cosine decay
        optimizer, lambda s: min(1, (s + 1) / 10) * 0.5 * (1 + math.cos(math.pi * s / len(batches))))
    base = model.get_base_model()
    model.train()
    losses, start = [], time.time()
    for batch in batches:
        width = max(len(ids) for ids, _ in batch)
        ids = torch.tensor([[tokenizer.pad_token_id] * (width - len(i)) + i for i, _ in batch], device=DEVICE)
        labels = torch.tensor([[-100] * (width - len(l)) + l for _, l in batch], device=DEVICE)
        mask = (ids != tokenizer.pad_token_id).long()
        hidden = base.model(input_ids=ids, attention_mask=mask, position_ids=(mask.cumsum(1) - 1).clamp(min=0)).last_hidden_state
        targets, keep = labels[:, 1:], labels[:, 1:] != -100
        logits = base.lm_head(hidden[:, :-1][keep]).float()  # vocabulary scores only on the answer tokens
        loss = torch.nn.functional.cross_entropy(logits, targets[keep])
        if math.isfinite(loss.item()):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        schedule.step()
        optimizer.zero_grad(set_to_none=True)
        losses.append(loss.item())
        if DEVICE == "mps" and len(losses) % 10 == 0:
            torch.mps.empty_cache()
    print(f"{len(examples):,} examples · {len(batches)} steps · {(time.time() - start) / 60:.1f} min")
    return model, losses


COLORS = {"1. Zero-shot": MUTED, "2. Fine-tuned (1,000)": ORANGE, "3. Fine-tuned (3,000)": BLUE}


def plot(losses: dict):
    """The training loss of each fine-tuning run, and the exact match of every approach so far."""
    fig, (loss_ax, score_ax) = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [1, 1.5]})
    for name, values in losses.items():
        loss_ax.plot(pd.Series(values).rolling(10, min_periods=1).mean(), color=COLORS[name], linewidth=2, label=name)
    loss_ax.set(title="Training loss", xlabel="step", ylabel="loss (log scale)", yscale="log")
    loss_ax.legend()

    groups = ["overall", "single", "missing_info", "multiple"]
    width = 0.8 / len(results)
    for i, (name, values) in enumerate(results.items()):
        xs = np.arange(len(groups)) + (i - (len(results) - 1) / 2) * width
        heights = [values[g] for g in groups]
        score_ax.bar(xs, heights, width=width * 0.92, color=COLORS[name], label=name)
        for x, h in zip(xs, heights):
            score_ax.text(x, h + 0.01, f"{h:.0%}", ha="center", va="bottom", fontsize=8)
    score_ax.set_xticks(range(len(groups)), groups)
    score_ax.set(title="Exact match on validation", ylim=(0, 1.1))
    score_ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
    score_ax.grid(axis="x", visible=False)
    score_ax.legend(loc="upper left", ncol=len(results))
    plt.tight_layout()
    plt.show()

In [ ]:
del model
model, loss_1k = fine_tune(train_pool[:1000])
results["2. Fine-tuned (1,000)"] = score(predict(model, validation))
losses = {"2. Fine-tuned (1,000)": loss_1k}

plot(losses)
pd.DataFrame(results).T.round(3)

Fine-tuning on 1,000 examples turns a model that cannot call tools into one that gets many simple requests right.

# Impact of Using More Training Data

The same fine-tuning on **3,000** examples. Set `MORE_DATA = len(train_pool)` to use all 14,700 (about 20 minutes
more on a T4).

In [ ]:
MORE_DATA = 3000

del model
model, losses["3. Fine-tuned (3,000)"] = fine_tune(train_pool[:MORE_DATA])
results["3. Fine-tuned (3,000)"] = score(predict(model, validation))

plot(losses)
pd.DataFrame(results).T.round(3)

More data reaches a lower loss and a much higher score in every category, most of all on requests with missing
values and with several calls.

The 3,000-example model writes a submission for the dev leaderboard:

In [ ]:
predictions = predict(model, dev)
empty = {"tool": "", "parameters": {}}  # an unreadable answer still needs an entry (it scores as wrong)
Path("predictions.json").write_text(json.dumps({k: v if v is not None else empty for k, v in predictions.items()}, ensure_ascii=False))
with zipfile.ZipFile("submission.zip", "w") as archive:
    archive.write("predictions.json")
print(f"{len(predictions):,} predictions → submission.zip")

# Ideas to Go Further

- Try FunctionGemma 270M, which was pre-trained for function calling
- Fine-tune a larger model. Remember it must be ≤ 4B parameters
- Generate more synthetic data and use more quality data with hard-mined examples
- Knowledge distillation
- Few-shot prompting
- Self-consistency and inference-time strategies
- Test-time scaling

**It's your world... Try → Evaluate → Improve → Repeat**